# PinkMap - Notebook 2: Clean, Explore, and Baseline

**What this does (Weeks 5 and 6 of the class):**
1. Loads the spreadsheet(s) your collection notebook made
2. **Cleans** the data (removes broken or repeated rows)
3. Adds helpful new columns (**features**), like "how much slower than usual?" and "is it rush hour?"
4. **Splits** the data into a training part, a checking part, and a test part
5. Makes **charts** to find patterns
6. Builds a **baseline** guess so the real model later has something to beat

**How to use it:** click **Runtime -> Run all**. Run it again whenever you have collected more data.


## Step 1. Load the data

Automatically use `data/auto_collected.csv` when run from a repository clone. Otherwise mount the Colab Google Drive folder or request a CSV upload.


In [ ]:
import os
import glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Works in Colab with Google Drive OR from a local clone of the repository.
USING_DRIVE = False
local_csv = Path("data/auto_collected.csv")
if local_csv.exists():
    SAVE_FOLDER = str(local_csv.parent.resolve())
    print("Using the repository's historical traffic CSV:", SAVE_FOLDER)
else:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        SAVE_FOLDER = "/content/drive/MyDrive/PinkMap"
        USING_DRIVE = True
        print("Google Drive connected.")
    except Exception:
        SAVE_FOLDER = str(Path.cwd() / "data")
        print("Drive unavailable; looking for CSV files here:", SAVE_FOLDER)

os.makedirs(SAVE_FOLDER, exist_ok=True)
UPLOAD_MORE = False  # In Colab, True opens an optional CSV upload dialogue.
csv_files = sorted(glob.glob(SAVE_FOLDER + "/*.csv"))

if not csv_files or UPLOAD_MORE:
    try:
        from google.colab import files
        print("Choose CSV file(s) to upload.")
        uploaded = files.upload()
        for name, content in uploaded.items():
            with open(os.path.join(SAVE_FOLDER, name), "wb") as f:
                f.write(content)
    except ImportError as exc:
        raise FileNotFoundError("Put auto_collected.csv in the repository data/ folder") from exc
    csv_files = sorted(glob.glob(SAVE_FOLDER + "/*.csv"))

if not csv_files:
    raise FileNotFoundError("No CSV files found in " + SAVE_FOLDER)
print("CSV files found:", len(csv_files))
for f in csv_files:
    print("   ", os.path.basename(f))
raw = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
print("Total rows loaded:", len(raw))
raw.head()


## Step 2. Quality check (look before you clean)
How many values are missing? Are there repeated rows?

In [ ]:
print("Missing values in each column:")
print(raw.isna().sum())

print("\nRepeated rows (same trip at the same time):",
      raw.duplicated(subset=["route_id", "collected_at"]).sum())

## Step 3. Clean the data
We remove rows that cannot be trusted, and we write down how many we removed and why. Writing this down is part of the class requirement (data cleaning and quality assessment).

In [ ]:
df = raw.copy()
start_rows = len(df)
log = []

# 1) Turn the time text into a real date and time
df["collected_at"] = pd.to_datetime(df["collected_at"], errors="coerce")

# 2) Make sure number columns are numbers
for col in ["distance_miles", "traffic_seconds", "typical_seconds"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 3) Remove rows missing something important
before = len(df)
df = df.dropna(subset=["collected_at", "distance_miles", "traffic_seconds", "typical_seconds"])
log.append(("Missing time, distance, traffic time, or typical time", before - len(df)))

# 4) Remove rows with impossible values
before = len(df)
df = df[(df["distance_miles"] > 0) & (df["traffic_seconds"] > 0) & (df["typical_seconds"] > 0)]
log.append(("Zero or negative values", before - len(df)))

# 5) Remove repeated rows
before = len(df)
df = df.drop_duplicates(subset=["route_id", "collected_at"])
log.append(("Repeated rows", before - len(df)))

df = df.sort_values("collected_at").reset_index(drop=True)

print("Rows before cleaning:", start_rows)
for reason, count in log:
    print(f"  removed {count:3d}  -  {reason}")
print("Rows after cleaning: ", len(df))

## Step 4. Add helpful new columns (features)
These help the computer see patterns.

| New column | Meaning |
|---|---|
| `traffic_delay_seconds` | traffic time minus typical time (positive = slower than usual) |
| `traffic_status` | slower / faster / normal (more than 30 seconds off counts) |
| `hour`, `day_of_week`, `is_weekend` | when the trip was measured |
| `time_period` | morning, afternoon, evening, or night |
| `rush_hour` | 1 if it is 7-9 AM or 4-6 PM, otherwise 0 |
| `avg_speed_mph` | distance divided by travel time |

In [ ]:
THRESHOLD_SECONDS = 30   # how far from "usual" counts as slower or faster

df["traffic_delay_seconds"] = df["traffic_seconds"] - df["typical_seconds"]

df["traffic_status"] = np.where(df["traffic_delay_seconds"] > THRESHOLD_SECONDS, "slower",
                        np.where(df["traffic_delay_seconds"] < -THRESHOLD_SECONDS, "faster", "normal"))

df["hour"] = df["collected_at"].dt.hour
df["day_of_week"] = df["collected_at"].dt.day_name()
df["is_weekend"] = (df["collected_at"].dt.dayofweek >= 5).astype(int)

df["time_period"] = np.where(df["hour"] < 12, "morning",
                     np.where(df["hour"] < 16, "afternoon",
                     np.where(df["hour"] < 20, "evening", "night")))

df["rush_hour"] = df["hour"].isin([7, 8, 16, 17]).astype(int)

df["avg_speed_mph"] = df["distance_miles"] / (df["traffic_seconds"] / 3600)

df.head()

## Step 5. Split complete collection runs: train / validation / test

- **Train (~70% of runs)**: oldest collections, used to learn patterns.
- **Validation (~10%)**: later collections, used to compare settings.
- **Test (~20%)**: newest collections, reserved for a final check.

**Why group runs?** Each collector execution requests multiple routes. Splitting individual rows could put results from the same collection run into both training and testing. We keep all rows from one run together, infer a new run after a gap over 10 minutes **or** when route IDs restart, and order runs chronologically.


In [ ]:
# Infer independent runs before splitting (supports 15-route historical data and 50-route future data).
df = df.sort_values(["collected_at", "route_id"], kind="mergesort").reset_index(drop=True)
gap_seconds = df["collected_at"].diff().dt.total_seconds().fillna(999999)
route_reset = df["route_id"].diff().fillna(-1) <= 0
df["collection_run"] = ((gap_seconds > 600) | route_reset).cumsum().astype(int)
run_ids = sorted(df["collection_run"].unique())
n_runs = len(run_ids)
if n_runs < 10:
    raise ValueError("Collect at least 10 independent runs before making a train/validation/test split")
train_cut = max(1, int(n_runs * 0.70))
val_cut = max(train_cut + 1, int(n_runs * 0.80))
train = df[df["collection_run"].isin(run_ids[:train_cut])].copy()
val = df[df["collection_run"].isin(run_ids[train_cut:val_cut])].copy()
test = df[df["collection_run"].isin(run_ids[val_cut:])].copy()
assert set(train.collection_run).isdisjoint(val.collection_run)
assert set(train.collection_run).isdisjoint(test.collection_run)
assert set(val.collection_run).isdisjoint(test.collection_run)
print(f"{n_runs} collection runs -> train {train.collection_run.nunique()}, validation {val.collection_run.nunique()}, test {test.collection_run.nunique()}")
print(f"Rows -> train {len(train)}, validation {len(val)}, test {len(test)}")
print("Date boundaries:", train.collected_at.min(), "to", test.collected_at.max())


## Step 6. Describe the data (Week 6)
Basic numbers, like the ones on your presentation slides, but now calculated from real data.

In [ ]:
summary = {
    "Route observations (rows)": len(df),
    "Different trips (routes)": df["route_id"].nunique(),
    "Different collection hours": df["collected_at"].dt.strftime("%Y-%m-%d %H").nunique(),
    "Average distance (miles)": round(df["distance_miles"].mean(), 2),
    "Shortest / longest trip (miles)": f"{df['distance_miles'].min():.2f} / {df['distance_miles'].max():.2f}",
    "Average traffic travel time (minutes)": round(df["traffic_seconds"].mean() / 60, 2),
    "Average size of delay (seconds, ignoring + or -)": round(df["traffic_delay_seconds"].abs().mean(), 1),
    "Correlation: distance vs traffic time": round(df["distance_miles"].corr(df["traffic_seconds"]), 3),
}
for key, value in summary.items():
    print(f"{key}: {value}")

print("\nHow many rows were slower / faster / normal:")
print(df["traffic_status"].value_counts())

## Step 7. Charts: look for patterns

In [ ]:
# Chart 1: traffic time vs typical time (dots above the line = slower than usual)
plt.figure(figsize=(6, 5))
plt.scatter(df["typical_seconds"] / 60, df["traffic_seconds"] / 60, alpha=0.6)
top = max(df["typical_seconds"].max(), df["traffic_seconds"].max()) / 60
plt.plot([0, top], [0, top], linestyle="--", color="gray")
plt.xlabel("Typical travel time (minutes)")
plt.ylabel("Travel time with traffic (minutes)")
plt.title("Traffic time vs typical time\n(above the dashed line = slower than usual)")
plt.tight_layout()
plt.show()

In [ ]:
# Chart 2: average delay by hour of day
by_hour = df.groupby("hour")["traffic_delay_seconds"].mean()
plt.figure(figsize=(8, 4))
by_hour.plot(kind="bar", color="#b5707f")
plt.axhline(0, color="gray")
plt.xlabel("Hour of day (24-hour clock)")
plt.ylabel("Average delay (seconds)")
plt.title("Is traffic worse at certain hours?")
plt.tight_layout()
plt.show()

In [ ]:
# Chart 3: average delay for each trip
by_route = df.groupby("route_id")["traffic_delay_seconds"].mean()
plt.figure(figsize=(8, 4))
by_route.plot(kind="bar", color="#1f2f4d")
plt.axhline(0, color="gray")
plt.xlabel("Trip number (route_id)")
plt.ylabel("Average delay (seconds)")
plt.title("Which trips are usually slower than normal?")
plt.tight_layout()
plt.show()

## Step 8. Baseline models
A **baseline** is a simple guess. Later, your real model must beat it.

- **Baseline A:** guess that traffic time = Mapbox's typical time
- **Baseline B:** a small straight-line model that uses distance and typical time

**MAE** = the average mistake in seconds. **RMSE** = like MAE, but big mistakes count extra. Smaller is better.

In [ ]:
from sklearn.linear_model import LinearRegression

def errors(actual, guess):
    e = np.array(actual) - np.array(guess)
    return float(np.mean(np.abs(e))), float(np.sqrt(np.mean(e ** 2)))

target = "traffic_seconds"
inputs = ["distance_miles", "typical_seconds"]

if len(train) < 3 or len(test) < 1:
    print("Not enough data yet. Collect more runs and run this notebook again.")
else:
    # Baseline A
    a_mae, a_rmse = errors(test[target], test["typical_seconds"])

    # Baseline B
    lin = LinearRegression().fit(train[inputs], train[target])
    b_mae, b_rmse = errors(test[target], lin.predict(test[inputs]))

    results = pd.DataFrame({
        "Model": ["A: guess = typical time", "B: linear (distance + typical time)"],
        "Test MAE (sec)": [round(a_mae, 1), round(b_mae, 1)],
        "Test RMSE (sec)": [round(a_rmse, 1), round(b_rmse, 1)],
    })
    print(f"Test rows used: {len(test)}")
    display(results)

## Step 9. Save the cleaned files
These files go in a `processed` folder. The next notebook (the real model) will use them.

In [ ]:
out = os.path.join(SAVE_FOLDER, "processed")
os.makedirs(out, exist_ok=True)

df.to_csv(out + "/clean_data.csv", index=False)
train.to_csv(out + "/train.csv", index=False)
val.to_csv(out + "/val.csv", index=False)
test.to_csv(out + "/test.csv", index=False)
print("Saved clean_data.csv, train.csv, val.csv, test.csv to:", out)

if not USING_DRIVE:
    try:
        from google.colab import files
    except ImportError:
        pass  # When running locally, inspect the data/processed folder.
    else:
        files.download(out + "/clean_data.csv")

- **Where the data came from:** OpenStreetMap (places) and the Mapbox Directions API (traffic and typical travel times).
- **What we cleaned:** see the cleaning log in Step 3.
- **What we added:** delay, status, hour, day, rush hour, and speed (Step 4).
- **How we split:** by time, 70/10/20 (Step 5).
- **What we found:** see the charts and numbers in Steps 6-7.
- **Baseline results:** see the table in Step 8. With a small dataset, treat the numbers as a proof of concept, not a final result.

**Deployment note:** Notebook 2 produces the exploratory analysis and baseline. To retrain/export the experimental prediction model for the HTML app, use `train_model.py` or `PinkMap_3_Train_Deploy_Model.ipynb` in the full repository. Predictions for new routes remain unvalidated until sufficient 50-route data is collected.
